# M3L2 E01 - LCEL Chain: componer con `|` (Resolution)

## Este notebook necesita API key de OpenAI


## Mapa de conceptos

| Concepto | Pregunta guia | En este notebook |
|---|---|---|
| LLM wrapper | Como encapsulo el modelo? | `ChatOpenAI(model=..., temperature=0)` |
| OutputParser | Como normalizo la salida? | `StrOutputParser()` |
| LCEL | Como conecto los componentes? | `prompt \| llm \| parser` |
| Reemplazo de modelo | Puedo cambiar el modelo sin tocar el resto? | Si, cambiando solo la variable `llm` |


In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")
print("API key cargada.")


## Bloque 1 - Sin LangChain


In [ ]:
from openai import OpenAI

client = OpenAI()

def answer_without_langchain(question: str) -> str:
    messages = [
        {"role": "system", "content": "Eres un asistente util. Responde de forma concisa."},
        {"role": "user", "content": question},
    ]
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0,
    )
    return response.choices[0].message.content


respuesta = answer_without_langchain("Cual es la capital de Francia?")
print(f"Respuesta: {respuesta}")


## Bloque 2 - Con LangChain: LCEL chain


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

print(f"LLM configurado: {llm.model_name}")


In [ ]:
# TODO 1: crear el PromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente util. Responde de forma concisa."),
    ("human", "{question}"),
])

print(f"Prompt creado: {type(prompt).__name__}")
print(f"Variables: {prompt.input_variables}")


In [ ]:
# TODO 2: componer la chain con LCEL
chain = prompt | llm | parser

print(f"Chain creada: {type(chain).__name__}")


In [ ]:
# TODO 3: invocar la chain
respuesta = chain.invoke({"question": "Cual es la capital de Francia?"})
print(f"Respuesta: {respuesta}")
print(f"Tipo de respuesta: {type(respuesta)}")
print("StrOutputParser garantiza que la respuesta es un string simple")


## Bloque 3 - El modelo es reemplazable


In [ ]:
llm_v2 = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)
chain_v2 = prompt | llm_v2 | parser

respuesta_v2 = chain_v2.invoke({"question": "Cual es la capital de Francia?"})
print(f"Con temperatura 0.5: {respuesta_v2}")
print()
print("El prompt y el parser son los mismos. Solo cambiamos el componente LLM.")
print("Eso es 'reemplazo de componentes' (Lecture - Seccion 7.2).")


## Checks


In [ ]:
def run_checks():
    assert prompt is not None
    assert chain is not None
    test_response = chain.invoke({"question": "Di solo la palabra 'test'"})
    assert isinstance(test_response, str)
    assert len(test_response) > 0
    r1 = chain.invoke({"question": "Cual es 2 + 2? Responde solo el numero"})
    assert "4" in r1
    print("M3L2 E01 Resolution checks passed")


run_checks()


## Cierre

| Sin LangChain | Con LangChain (LCEL) |
|---|---|
| Llamada imperativa: `client.chat.completions.create(...)` | Composicion declarativa: `prompt \| llm \| parser` |
| Modelo hardcodeado en cada funcion. | Modelo configurado en un objeto. |
| Extraer texto: `response.choices[0].message.content` | Parser lo hace automaticamente. |
| Cambiar modelo = modificar muchos archivos. | Cambiar modelo = reemplazar el objeto `llm`. |
